In [1]:
import torch
from PIL import Image
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torchvision.transforms as transforms

# Transformación
transform_val = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


# Modelo
class MultiEfficientNet(torch.nn.Module):
    def __init__(self, base_model, num_classes_d, num_classes_o):
        super().__init__()
        self.features = torch.nn.Sequential(*list(base_model.children())[:-1])
        self.pool = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = torch.nn.Flatten()
        self.classifier_d = torch.nn.Linear(base_model.classifier[1].in_features, num_classes_d)
        self.classifier_o = torch.nn.Linear(base_model.classifier[1].in_features, num_classes_o)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.flatten(x)
        out_d = self.classifier_d(x)
        out_o = self.classifier_o(x)
        return out_d, out_o


# Clases
idx_to_class_d = {i: i for i in range(8)}
idx_to_class_o = {i: i for i in range(8)}

# Cargar modelo
device = torch.device("cpu")  # Forzar CPU por seguridad
base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = MultiEfficientNet(base, len(idx_to_class_d), len(idx_to_class_o))
model.load_state_dict(torch.load("modelo_ductos_multitarea_efnet.pth", map_location=device))
model.to(device)
model.eval()

# Ruta manual (modifícala)
ruta_img = "img_predic.jpg"

# Predicción
img = Image.open(ruta_img).convert("RGB")
img_tensor = transform_val(img).unsqueeze(0).to(device)

with torch.no_grad():
    out_d, out_o = model(img_tensor)
    pred_d = torch.argmax(out_d, dim=1).item()
    pred_o = torch.argmax(out_o, dim=1).item()

d_total = idx_to_class_d[pred_d]
o_total = idx_to_class_o[pred_o]
v_total = max(d_total - o_total, 0)

print(f"\n📸 Imagen: {ruta_img}")
print(f"🔢 Ductos totales (dX): {d_total if d_total < 7 else '7+'}")
print(f"✅ Ductos ocupados (oX): {o_total if o_total < 7 else '7+'}")
print(f"⬜ Ductos vacíos     : {v_total}")

/Users/dilanacosta/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/dilanacosta/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📸 Imagen: img_predic.jpg
🔢 Ductos totales (dX): 4
✅ Ductos ocupados (oX): 3
⬜ Ductos vacíos     : 1
